In [1]:
import pandas as pd
import numpy as np

clients = pd.read_csv("../data/raw/SaintHonore_clients.csv")
boutiques = pd.read_csv("../data/raw/SaintHonore_boutiques.csv")

clients.head()

,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,montant_total_centimes,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m
0,CT80059,BT15,4.0,4.0,1468.0,587200.0,0,0,aucune,en ligne,USA,725.0,oui
1,CT80578,BT28,80.0,3.0,1235.0,370500.0,1,1,argent,boutique,France,1479.0,oui
2,CT80256,BT27,43.0,1.0,1814.0,181400.0,0,2,or,boutique,France,2307.0,oui
3,CT80098,BT16,1.0,1.0,1795.0,179500.0,0,0,or,boutique,Italie,1820.0,non
4,CT80362,BT31,51.0,3.0,2022.0,606600.0,0,0,argent,boutique,Chine,1201.0,oui


In [2]:
pd.set_option("display.max_columns", None)  # afficher toutes les colonnes

print("Taille :", clients.shape)
print()
print(clients.dtypes)
print()
print("Valeurs manquantes :")
print(clients.isna().sum())

Taille : (770, 13)

client_id                         str
boutique_principale               str
anciennete_mois               float64
nb_achats_12m                 float64
panier_moyen_eur              float64
montant_total_centimes        float64
retours                         int64
demandes_sav                    int64
carte_fidelite                    str
canal_prefere                     str
pays_residence                    str
montant_prochain_achat_eur    float64
est_revenu_6m                     str
dtype: object

Valeurs manquantes :
client_id                      0
boutique_principale            0
anciennete_mois                0
nb_achats_12m                  0
panier_moyen_eur              22
montant_total_centimes         0
retours                        0
demandes_sav                   0
carte_fidelite                22
canal_prefere                  0
pays_residence                 0
montant_prochain_achat_eur     0
est_revenu_6m                  0
dtype: int64


In [9]:
for col in ["boutique_principale", "carte_fidelite", "canal_prefere", "pays_residence", "est_revenu_6m"]:
    print(f"--- {col} ---")
    print(clients[col].value_counts(dropna=False))
    print()

print("Lignes entièrement dupliquées :", clients.duplicated().sum())
print("client_id en double :", clients["client_id"].duplicated().sum())

--- boutique_principale ---
boutique_principale
BT11    42
BT30    41
BT20    40
BT16    38
BT15    37
BT23    36
BT32    34
BT13    34
BT10    34
BT26    34
BT25    34
BT21    32
BT12    32
BT28    31
BT14    30
BT22    30
BT24    29
BT29    29
BT33    29
BT31    26
BT17    26
BT19    25
BT18    25
BT27    22
Name: count, dtype: int64

--- carte_fidelite ---
carte_fidelite
aucune    367
argent    265
or        116
NaN        22
Name: count, dtype: int64

--- canal_prefere ---
canal_prefere
boutique    440
en ligne    174
mixte       156
Name: count, dtype: int64

--- pays_residence ---
pays_residence
France    254
Chine     148
USA       116
Italie     72
EAU        71
Corée      64
Suisse     45
Name: count, dtype: int64

--- est_revenu_6m ---
est_revenu_6m
non    454
oui    316
Name: count, dtype: int64

Lignes entièrement dupliquées : 70
client_id en double : 70


In [10]:
avant = len(clients)
clients = clients.drop_duplicates()
print("Lignes supprimées :", avant - len(clients))
print("Lignes restantes :", len(clients))
print("client_id encore en double :", clients["client_id"].duplicated().sum())

Lignes supprimées : 70
Lignes restantes : 700
client_id encore en double : 0


In [11]:
clients.isna().sum()

client_id                      0
boutique_principale            0
anciennete_mois                0
nb_achats_12m                  0
panier_moyen_eur              21
montant_total_centimes         0
retours                        0
demandes_sav                   0
carte_fidelite                21
canal_prefere                  0
pays_residence                 0
montant_prochain_achat_eur     0
est_revenu_6m                  0
dtype: int64

In [12]:
clients.describe().round(1)

,anciennete_mois,nb_achats_12m,panier_moyen_eur,montant_total_centimes,retours,demandes_sav,montant_prochain_achat_eur
count,700.0,700.0,679.0,700.0,700.0,700.0,700.0
mean,30.8,3.7,2622.8,1267470.6,0.5,0.4,1972.0
std,24.3,2.9,1895.2,2052227.1,0.8,0.6,1135.3
min,1.0,0.0,-200.0,29000.0,0.0,0.0,90.0
25%,12.0,2.0,1503.5,290900.0,0.0,0.0,1254.5
50%,23.0,3.0,2076.0,530850.0,0.0,0.0,1692.5
75%,49.0,5.0,2895.0,927675.0,1.0,1.0,2334.2
max,116.0,17.0,12685.0,11416500.0,5.0,3.0,6708.0


In [13]:
for col in ["anciennete_mois", "nb_achats_12m"]:
    non_entiers = (clients[col] != clients[col].round()).sum()
    print(col, ": valeurs non entières =", non_entiers)

anciennete_mois : valeurs non entières = 0
nb_achats_12m : valeurs non entières = 0


In [14]:
print("Paniers <= 0 :", (clients["panier_moyen_eur"] <= 0).sum())
display(clients[clients["panier_moyen_eur"] <= 0])

print("Clients avec 0 achat :", (clients["nb_achats_12m"] == 0).sum())
display(clients[clients["nb_achats_12m"] == 0].head(10))

Paniers <= 0 : 3


,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,montant_total_centimes,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m
312,CT80187,BT23,21.0,1.0,-200.0,241400.0,1,0,argent,boutique,Chine,1486.0,non
477,CT80253,BT23,10.0,7.0,-200.0,3766000.0,0,0,NaN,boutique,Italie,4163.0,oui
525,CT80157,BT30,19.0,3.0,-200.0,366900.0,0,0,aucune,boutique,Italie,1694.0,non


Clients avec 0 achat : 3


,client_id,boutique_principale,anciennete_mois,nb_achats_12m,panier_moyen_eur,montant_total_centimes,retours,demandes_sav,carte_fidelite,canal_prefere,pays_residence,montant_prochain_achat_eur,est_revenu_6m
35,CT80389,BT14,59.0,0.0,2276.0,455200.0,1,0,NaN,boutique,Italie,1349.0,oui
229,CT80352,BT26,16.0,0.0,1948.0,389600.0,1,1,or,en ligne,France,1311.0,non
603,CT80287,BT17,19.0,0.0,2768.0,553600.0,0,0,aucune,mixte,France,2116.0,non


In [15]:
print(455200 / 2276 / 100)
print(389600 / 1948 / 100)
print(553600 / 2768 / 100)

2.0
2.0
2.0


In [16]:
calcul = clients["nb_achats_12m"] * clients["panier_moyen_eur"] * 100
ecart = (clients["montant_total_centimes"] - calcul).abs()

print("Lignes où la relation est vraie :", (ecart < 1).sum())
print("Lignes où elle est fausse :", (ecart >= 1).sum())
print("Lignes impossibles à vérifier (panier NaN) :", ecart.isna().sum())

Lignes où la relation est vraie : 673
Lignes où elle est fausse : 6
Lignes impossibles à vérifier (panier NaN) : 21


In [17]:
display(clients[ecart >= 1][["client_id", "nb_achats_12m", "panier_moyen_eur", "montant_total_centimes"]])

,client_id,nb_achats_12m,panier_moyen_eur,montant_total_centimes
35,CT80389,0.0,2276.0,455200.0
229,CT80352,0.0,1948.0,389600.0
312,CT80187,1.0,-200.0,241400.0
477,CT80253,7.0,-200.0,3766000.0
525,CT80157,3.0,-200.0,366900.0
603,CT80287,0.0,2768.0,553600.0


In [18]:
# 1. Nombre d'achats à 0 -> recalculé avec la formule
masque_0 = clients["nb_achats_12m"] == 0
clients.loc[masque_0, "nb_achats_12m"] = (
    clients.loc[masque_0, "montant_total_centimes"] / (clients.loc[masque_0, "panier_moyen_eur"] * 100)
)

# 2. Panier à -200 (code d'erreur) -> considéré comme manquant
clients.loc[clients["panier_moyen_eur"] <= 0, "panier_moyen_eur"] = np.nan

# 3. Tous les paniers manquants -> recalculés avec la formule
masque_nan = clients["panier_moyen_eur"].isna()
print("Paniers à recalculer :", masque_nan.sum())
clients.loc[masque_nan, "panier_moyen_eur"] = (
    clients.loc[masque_nan, "montant_total_centimes"] / (clients.loc[masque_nan, "nb_achats_12m"] * 100)
)

Paniers à recalculer : 24


In [19]:
calcul = clients["nb_achats_12m"] * clients["panier_moyen_eur"] * 100
ecart = (clients["montant_total_centimes"] - calcul).abs()

print("Lignes où la relation est vraie :", (ecart < 1).sum())
print("Lignes où elle est fausse :", (ecart >= 1).sum())
print("Lignes impossibles à vérifier (panier NaN) :", ecart.isna().sum())

Lignes où la relation est vraie : 700
Lignes où elle est fausse : 0
Lignes impossibles à vérifier (panier NaN) : 0
